# Adult Income Classification — TFX Machine Learning Pipeline

---

## 1. Project Description

### Dataset
The dataset used is **Adult Income (Census Income)** from the UCI Machine Learning Repository. This dataset contains US census data collected in 1994.

- **Source:** https://archive.ics.uci.edu/dataset/2/adult
- **Total records:** 48,842 rows
- **Numerical features:** `age`, `fnlwgt`, `education-num`, `capital-gain`, `capital-loss`, `hours-per-week`
- **Categorical features:** `workclass`, `education`, `marital-status`, `occupation`, `relationship`, `race`, `sex`, `native-country`
- **Label:** `income` (`<=50K` or `>50K`)

### Problem Statement
Predict whether a person's income **exceeds $50,000 per year** based on demographic and employment data. This is a **binary classification** problem.

### Machine Learning Solution
Build an **end-to-end ML pipeline** using TensorFlow Extended (TFX) with:
- Automated data validation (StatisticsGen, SchemaGen, ExampleValidator)
- Standardized preprocessing (Transform)
- Automated hyperparameter tuning (Tuner with KerasTuner)
- DNN model training (Trainer)
- Blessing-based evaluation (Evaluator)
- Deployment to TF Serving (Pusher)

### Performance Target
Binary Accuracy ≥ **0.80** on evaluation data.

### Data Processing Methods
- **Numerical features**: Z-score normalization using pandas preprocessing
- **Categorical features**: Label encoding using pandas preprocessing
- **Label**: Binary encoding (`<=50K` → 0, `>50K` → 1) in pandas
- **TFX Transform**: Pass-through (`tf.cast` only for type conversion)

### Model Architecture
Deep Neural Network (DNN):
- Embedding layers for categorical features
- Dense(128, ReLU) → Dropout(0.3) → Dense(64, ReLU) → Dropout(0.3) → Dense(1, Sigmoid)
- Optimizer: Adam
- Loss: Binary Crossentropy

### Evaluation Metrics
- **Binary Accuracy** (primary metric, threshold ≥ 0.80)
- **AUC-ROC**
- **Binary Crossentropy**


## 2. Setup and Installation

Run this cell only if using **Google Colab**. For Compute Engine, skip this cell because TFX is already installed in the conda environment.

In [1]:
# Make sure the tfx-submission conda environment is active before running this notebook
# conda activate tfx-submission

import sys
print(f"Python version: {sys.version}")
print("Make sure environment: tfx-submission (Python 3.9.15)")

Python version: 3.9.15 (main, Nov 24 2022, 14:31:59) 
[GCC 11.2.0]
Make sure environment: tfx-submission (Python 3.9.15)


## 3. Import Libraries

In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import tfx

from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Trainer,
    Evaluator,
    Pusher,
)
from tfx.components import Tuner
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental.latest_blessed_model_resolver import LatestBlessedModelResolver
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
import tensorflow_model_analysis as tfma

print('TFX version  :', tfx.__version__)
print('TF version   :', tf.__version__)

2026-05-14 05:49:52.965665: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 05:49:53.155624: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-14 05:49:53.155652: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-05-14 05:49:53.199275: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 05:49:54.245841: W tensorflow/stream_executor/platform/de

TFX version  : 1.11.0
TF version   : 2.10.1


## 4. Data Preparation

In [3]:
# Pipeline directory
PIPELINE_NAME     = 'adult-income-pipeline'
PIPELINE_ROOT     = os.path.join(os.getcwd(), PIPELINE_NAME)
DATA_ROOT         = os.path.join(os.getcwd(), 'data')
SERVING_MODEL_DIR = os.path.join(os.getcwd(), 'serving_model', 'adult_income_model')
TRANSFORM_MODULE  = os.path.join(os.getcwd(), 'transform_module.py')
TRAINER_MODULE    = os.path.join(os.getcwd(), 'trainer_module.py')
TUNER_MODULE      = os.path.join(os.getcwd(), 'tuner_module.py')

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(SERVING_MODEL_DIR, exist_ok=True)

print(f'Pipeline root : {PIPELINE_ROOT}')
print(f'Data root     : {DATA_ROOT}')

Pipeline root : /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline
Data root     : /home/sintiawati_putu04/mlops-tfx-submission/data


In [4]:
# Download and prepare Adult Income dataset from UCI
# ALL preprocessing is done in pandas (z-score, label encoding)
# Transform module only does pass-through (tf.cast)
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
url_test  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'

df_train = pd.read_csv(url_train, header=None, names=COLUMNS,
                       na_values='?', skipinitialspace=True)
df_test  = pd.read_csv(url_test,  header=None, names=COLUMNS,
                       na_values='?', skipinitialspace=True, skiprows=1)

# Remove trailing period from test set labels
df_test['income'] = df_test['income'].str.replace('.', '', regex=False)

# Merge and drop missing values
df = pd.concat([df_train, df_test], ignore_index=True)
df.dropna(inplace=True)

# Preprocessing: all done in pandas (not in TFX Transform)
# 1. Encode label: >50K -> 1, <=50K -> 0
df['income'] = (df['income'].str.strip() == '>50K').astype(int)

# 2. Encode categorical features ke integer
CATEGORICAL_FEATURES = [
    'workclass', 'education', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'native-country',
]
for col in CATEGORICAL_FEATURES:
    codes, _ = pd.factorize(df[col])
    df[col] = codes

# 3. Z-score normalization untuk numeric features
NUMERIC_FEATURES = [
    'age', 'fnlwgt', 'education-num',
    'capital-gain', 'capital-loss', 'hours-per-week',
]
for col in NUMERIC_FEATURES:
    mean = df[col].mean()
    std  = df[col].std()
    df[col] = (df[col] - mean) / std

print(f'Total data after preprocessing: {len(df)}')
print(f'Label distribution:\n{df["income"].value_counts()}')
print(f'\nSample data (all numeric):')
df.head()

Total data after preprocessing: 45222
Label distribution:
0    34014
1    11208
Name: income, dtype: int64

Sample data (all numeric):


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.034201,0,-1.062283,0,1.128740,0,0,0,0,0,0.142887,-0.218778,-0.078119,0,0
1,0.866407,1,-1.007427,0,1.128740,1,1,1,0,0,-0.146732,-0.218778,-2.326712,0,0
2,-0.041455,2,0.245281,1,-0.438117,2,2,0,0,0,-0.146732,-0.218778,-0.078119,0,0
3,1.093373,2,0.425848,2,-1.221545,1,2,1,1,0,-0.146732,-0.218778,-0.078119,0,0
4,-0.798006,2,1.407378,0,1.128740,1,3,2,1,1,-0.146732,-0.218778,-0.078119,1,0


In [5]:
# Save to CSV for ExampleGen
# Use csv/ subfolder for isolation from other files
import glob, json
csv_dir = os.path.join(DATA_ROOT, 'csv')
os.makedirs(csv_dir, exist_ok=True)
csv_path = os.path.join(csv_dir, 'adult_income.csv')
for old_csv in glob.glob(os.path.join(csv_dir, '*.csv')):
    os.remove(old_csv)
df.to_csv(csv_path, index=False)
print(f'Data saved to: {csv_path}')
print(f'Shape: {df.shape}')

# Save samples for testing prediction (without label)
test_samples = df.drop(columns=['income']).head(3).to_dict(orient='records')
samples_path = os.path.join(DATA_ROOT, 'test_samples.json')
with open(samples_path, 'w') as f:
    json.dump(test_samples, f)
print(f'Test samples saved to: {samples_path}')

Data saved to: /home/sintiawati_putu04/mlops-tfx-submission/data/csv/adult_income.csv
Shape: (45222, 15)
Test samples saved to: /home/sintiawati_putu04/mlops-tfx-submission/data/test_samples.json


## 5. InteractiveContext

Create an `InteractiveContext` as the runner for all TFX components interactively. All pipeline artifacts will be stored in the `adult-income-pipeline` folder.

In [6]:
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)
print(f'Pipeline artifacts will be saved at: {PIPELINE_ROOT}')

Pipeline artifacts will be saved at: /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline


## 6. ExampleGen

First pipeline component. `CsvExampleGen` reads CSV data and splits it into **train** (2/3) and **eval** (1/3) splits.

In [7]:
example_gen = CsvExampleGen(input_base=csv_dir)
context.run(example_gen)
print('ExampleGen complete!')
print('Artifacts:', example_gen.outputs['examples'].get())

ExampleGen complete!
Artifacts: [Artifact(artifact: id: 2
type_id: 14
uri: "/home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/CsvExampleGen/examples/2"
properties {
  key: "split_names"
  value {
    string_value: "[\"train\", \"eval\"]"
  }
}
custom_properties {
  key: "file_format"
  value {
    string_value: "tfrecords_gzip"
  }
}
custom_properties {
  key: "input_fingerprint"
  value {
    string_value: "split:single_split,num_files:1,total_bytes:6130182,xor_checksum:1778737803,sum_checksum:1778737803"
  }
}
custom_properties {
  key: "payload_format"
  value {
    string_value: "FORMAT_TF_EXAMPLE"
  }
}
custom_properties {
  key: "span"
  value {
    int_value: 0
  }
}
custom_properties {
  key: "state"
  value {
    string_value: "published"
  }
}
custom_properties {
  key: "tfx_version"
  value {
    string_value: "1.11.0"
  }
}
state: LIVE
, artifact_type: id: 14
name: "Examples"
properties {
  key: "span"
  value: INT
}
properties {
  key: "split_names"
  valu

## 7. StatisticsGen

Compute descriptive statistics of the data (mean, std, distribution, missing values) for each split.

In [8]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs['examples']
)
context.run(statistics_gen)
context.show(statistics_gen.outputs['statistics'])

## 8. SchemaGen

Automatically infer the data schema from statistics. The schema defines data types, value ranges, and domains for each feature.

In [9]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True,
)
context.run(schema_gen)
context.show(schema_gen.outputs['schema'])

,Type,Presence,Valency,Domain
Feature name,,,,
'age',FLOAT,required,,-
'capital-gain',FLOAT,required,,-
'capital-loss',FLOAT,required,,-
'education',INT,required,,-
'education-num',FLOAT,required,,-
'fnlwgt',FLOAT,required,,-
'hours-per-week',FLOAT,required,,-
'income',INT,required,,-
'marital-status',INT,required,,-


## 9. ExampleValidator

Validate data against the schema. Detects anomalies such as out-of-range values, wrong data types, or missing features.

In [10]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema'],
)
context.run(example_validator)
context.show(example_validator.outputs['anomalies'])

## 10. Transform

Pass-through transform. All preprocessing (z-score, label encoding, categorical encoding)
was already done in pandas in step 4. The Transform module only performs `tf.cast`
to adjust data types (float32 for numerical, int64 for categorical & label).

In [11]:
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=TRANSFORM_MODULE,
)
context.run(transform)
print('Transform complete!')

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying trainer_module.py -> build/lib
copying transform_module.py -> build/lib
installing to /tmp/tmpcv0zstrz
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmpcv0zstrz
copying build/lib/transform_module.py -> /tmp/tmpcv0zstrz
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpcv0zstrz/tfx_user_code_Transform-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3

/home/sintiawati_putu04/miniconda/envs/tfx-submission/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./adult-income-pipeline/_wheels/tfx_user_code_Transform-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3-none-any.whl
Processing ./adult-income-pipeline/_wheels/tfx_user_code_Transform-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3-none-any.whl
Processing ./adult-income-pipeline/_wheels/tfx_user_code_Transform-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3-none-any.whl
Instructions for updating:
Use ref() instead.


2026-05-14 05:50:38.622594: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-05-14 05:50:38.622702: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2026-05-14 05:50:38.622773: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (tfx-worker): /proc/driver/nvidia/version does not exist
2026-05-14 05:50:38.623819: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/Transform/transform_graph/6/.temp_path/tftransform_tmp/dd147ecdd6be437f948a707abeabef54/assets


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


Transform complete!


## 11. Tuner (Bonus)

Automatic hyperparameter tuning using **KerasTuner RandomSearch** with 5 trials. Tuned parameters:
- `units_1`: jumlah neuron layer 1
- `units_2`: jumlah neuron layer 2
- `dropout_rate`: dropout rate
- `learning_rate`: learning rate

In [12]:
tuner = Tuner(
    module_file=TUNER_MODULE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=500),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=100),
)
context.run(tuner)
print('Tuner complete!')

Trial 5 Complete [00h 00m 05s]
val_binary_accuracy: 0.8506249785423279

Best val_binary_accuracy So Far: 0.85546875
Total elapsed time: 00h 00m 26s
Results summary
Results in /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/.temp/7/adult_income_tuning
Showing 10 best trials
Objective(name="val_binary_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
units_1: 256
units_2: 64
dropout_rate: 0.2
learning_rate: 0.001
Score: 0.85546875

Trial 4 summary
Hyperparameters:
units_1: 128
units_2: 128
dropout_rate: 0.2
learning_rate: 0.001
Score: 0.8506249785423279

Trial 3 summary
Hyperparameters:
units_1: 128
units_2: 64
dropout_rate: 0.30000000000000004
learning_rate: 0.01
Score: 0.8500000238418579

Trial 2 summary
Hyperparameters:
units_1: 256
units_2: 64
dropout_rate: 0.4
learning_rate: 0.001
Score: 0.8473437428474426

Trial 1 summary
Hyperparameters:
units_1: 128
units_2: 64
dropout_rate: 0.2
learning_rate: 0.0001
Score: 0.8276562690734863
Tuner complete!


## 12. Trainer

Training model DNN.

In [13]:
trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=200),
)
context.run(trainer)
print('Trainer complete!')

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying trainer_module.py -> build/lib
copying transform_module.py -> build/lib
installing to /tmp/tmptx6rxb46
running install
running install_lib
copying build/lib/trainer_module.py -> /tmp/tmptx6rxb46
copying build/lib/transform_module.py -> /tmp/tmptx6rxb46
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmptx6rxb46/tfx_user_code_Trainer-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3.9.egg-info
runnin

/home/sintiawati_putu04/miniconda/envs/tfx-submission/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()


Processing ./adult-income-pipeline/_wheels/tfx_user_code_Trainer-0.0+b4020dd209215d4a1de60a4c52d37cc669b9ba7cb580fa06eb6353eddb1bbca1-py3-none-any.whl
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 workclass_xf (InputLayer)      [(None, 1)]          0           []                               
                                                                                                  
 education_xf (InputLayer)      [(None, 1)]          0           []                               
                                                                                                  
 marital-status_xf (InputLayer)  [(None, 1)]         0           []                               
                                                                                                  
 occupation_xf (InputLayer)     [(None, 

                                                                  'flatten_9[0][0]',              
                                                                  'flatten_10[0][0]',             
                                                                  'flatten_11[0][0]',             
                                                                  'flatten_12[0][0]',             
                                                                  'flatten_13[0][0]',             
                                                                  'flatten_14[0][0]',             
                                                                  'flatten_15[0][0]']             
                                                                                                  
 dense_3 (Dense)                (None, 128)          9088        ['concatenate_3[0][0]']          
                                                                                                  
 dropout_2

1000/1000 [==============================] - 4s 4ms/step - loss: 0.3125 - binary_accuracy: 0.8568 - auc: 0.9131 - val_loss: 0.3112 - val_binary_accuracy: 0.8555 - val_auc: 0.9150
INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/Trainer/model/8/Format-Serving/assets


INFO:tensorflow:Assets written to: /home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/Trainer/model/8/Format-Serving/assets


Trainer complete!


## 13. Resolver

Retrieve the previous blessed model as the evaluation baseline.

In [14]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id('Latest_blessed_model_resolver')

context.run(model_resolver)
print('Resolver complete!')

Resolver complete!


## 14. Evaluator

Evaluate the model using TFMA. The model receives a blessing if `binary_accuracy` ≥ 0.80.

In [15]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key='income',
            signature_name='serving_default',
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.80}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': 0.001},
                        )
                    )
                ),
                tfma.MetricConfig(class_name='AUC'),
                tfma.MetricConfig(class_name='BinaryCrossentropy'),
            ]
        )
    ]
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config,
)
context.run(evaluator)
context.show(evaluator.outputs['evaluation'])

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


In [16]:
# Check blessing status
blessing = evaluator.outputs['blessing'].get()[0]
print('Model blessing status:', blessing.get_string_custom_property('blessed'))

Model blessing status: 


## 15. Pusher

Push the model to the serving directory if it receives a blessing.

In [17]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
context.run(pusher)
print('Pusher complete!')
print('Model serving saved to:', pusher.outputs['pushed_model'].get())

Pusher complete!
Model serving tersimpan di: [Artifact(artifact: id: 20
type_id: 35
uri: "/home/sintiawati_putu04/mlops-tfx-submission/adult-income-pipeline/Pusher/pushed_model/11"
custom_properties {
  key: "name"
  value {
    string_value: "pushed_model:2026-05-14T05:52:28.309133"
  }
}
custom_properties {
  key: "producer_component"
  value {
    string_value: "Pusher"
  }
}
custom_properties {
  key: "pushed"
  value {
    int_value: 1
  }
}
custom_properties {
  key: "pushed_destination"
  value {
    string_value: "/home/sintiawati_putu04/mlops-tfx-submission/serving_model/adult_income_model/1778737948"
  }
}
custom_properties {
  key: "pushed_version"
  value {
    string_value: "1778737948"
  }
}
custom_properties {
  key: "state"
  value {
    string_value: "published"
  }
}
custom_properties {
  key: "tfx_version"
  value {
    string_value: "1.11.0"
  }
}
state: LIVE
name: "pushed_model:2026-05-14T05:52:28.309133"
, artifact_type: id: 35
name: "PushedModel"
base_type: MODEL

## 16. Pipeline Summary

All TFX components have been successfully executed:

| # | Component | Description |
|---|---|---|
| 1 | ExampleGen | ✅ Read and split CSV data |
| 2 | StatisticsGen | ✅ Compute data statistics |
| 3 | SchemaGen | ✅ Auto-generate data schema |
| 4 | ExampleValidator | ✅ Validate data against schema |
| 5 | Transform | ✅ Preprocess data |
| 6 | Tuner | ✅ Hyperparameter tuning (Bonus) |
| 7 | Trainer | ✅ Train DNN model |
| 8 | Resolver | ✅ Retrieve baseline model |
| 9 | Evaluator | ✅ Evaluate with TFMA |
| 10 | Pusher | ✅ Deploy to serving directory |

Model has been pushed to `serving_model/adult_income_model/` and is ready to be served using TensorFlow Serving.